# KNN Regressor — Interactive Dashboard (Notebook Version)

This notebook walks through **every plot** from the Streamlit dashboard step by step,  
with detailed comments explaining what each line of code does and *why*.

> **Synthetic dataset:** 2 features → 1 continuous target  
> **Goal:** Understand how changing K changes the model's fit, residuals, and R² score.

---


## 0 · Install & Import Libraries

In [ ]:
%pip install numpy pandas matplotlib scikit-learn
# All standard libraries — no extra installs needed beyond these 4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap

from sklearn.neighbors import KNeighborsRegressor   # the model we are studying
from sklearn.preprocessing import StandardScaler    # scale features before KNN
from sklearn.metrics import mean_squared_error, r2_score  # evaluation metrics

import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully ✓")

---
## 1 · Global Settings

We fix a colour palette and matplotlib style here so every plot looks consistent.  
Using dark colours makes the prediction lines pop visually.


In [ ]:
# ── Colour palette ────────────────────────────────────────────────────────────
# These hex codes are used throughout every plot below.
BG      = "#0d0d14"   # figure / canvas background  (near-black)
PANEL   = "#12121e"   # axes background             (slightly lighter)
GRID    = "#1e1e30"   # grid lines
ACCENT1 = "#a78bfa"   # violet  → main KNN prediction line
ACCENT2 = "#60a5fa"   # blue    → scatter data points
ACCENT3 = "#34d399"   # green   → residuals
ACCENT4 = "#f97316"   # orange  → selected-K highlight / zero line
TEXT    = "#e8e8f0"   # axis labels and tick labels
SUBTEXT = "#606090"   # minor annotations

# ── Apply to all future matplotlib figures ────────────────────────────────────
# rcParams is a global dictionary that controls matplotlib defaults.
# Setting it once here means every plt.figure() call inherits these values.
plt.rcParams.update({
    "figure.facecolor" : BG,
    "axes.facecolor"   : PANEL,
    "axes.edgecolor"   : GRID,
    "axes.labelcolor"  : TEXT,
    "xtick.color"      : SUBTEXT,
    "ytick.color"      : SUBTEXT,
    "text.color"       : TEXT,
    "grid.color"       : GRID,
    "grid.linewidth"   : 0.6,
    "font.family"      : "monospace",
    "axes.titlesize"   : 11,
    "axes.labelsize"   : 9,
})

print("Global style applied ✓")

---
## 2 · Generate Synthetic Dataset

We create **300 samples** with **2 input features** (X1, X2) and **1 target y**.

### Why this particular formula?
`y = sin(X1) · cos(X2) + 0.5·X1 − 0.3·X2² + noise`

- It is **non-linear** → KNN with small K can capture the wiggles; large K over-smooths them.  
- It depends on **both features** → justifies having 2 features.  
- Adding Gaussian noise simulates real-world measurement uncertainty.


In [ ]:
# ── Parameters you can freely change ─────────────────────────────────────────
N_SAMPLES   = 300     # total data points
NOISE_LEVEL = 1.5     # standard deviation of added Gaussian noise
SEED        = 42      # for reproducibility

# ── Generate ──────────────────────────────────────────────────────────────────
rng = np.random.RandomState(SEED)

# Two independent features, both uniform in [-3, 3]
X1 = rng.uniform(-3, 3, N_SAMPLES)
X2 = rng.uniform(-3, 3, N_SAMPLES)

# Non-linear target: sin-cos interaction + linear X1 term + quadratic X2 term
y  = (np.sin(X1) * np.cos(X2)
      + 0.5  * X1
      - 0.3  * X2**2
      + NOISE_LEVEL * rng.randn(N_SAMPLES))   # Gaussian noise

# Stack into an (N, 2) feature matrix — standard sklearn input format
X = np.column_stack([X1, X2])

# Quick sanity check
print(f"X shape : {X.shape}")   # (300, 2)
print(f"y shape : {y.shape}")   # (300,)
print(f"y range : [{y.min():.2f},  {y.max():.2f}]")

---
## 3 · Feature Scaling with StandardScaler

KNN is a **distance-based** algorithm.  
If Feature 1 ranges [0, 1000] and Feature 2 ranges [0, 1], then  
distances are dominated by Feature 1 and Feature 2 is effectively ignored.

`StandardScaler` transforms each feature to **mean = 0, std = 1** so both  
features contribute equally to the distance calculation.

**Formula:**  `x_scaled = (x − μ) / σ`


In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)   # fit learns μ and σ; transform applies them

print(f"Before scaling — X1 mean: {X[:,0].mean():.3f},  std: {X[:,0].std():.3f}")
print(f"After  scaling — X1 mean: {X_scaled[:,0].mean():.3f},  std: {X_scaled[:,0].std():.3f}")
print()
print(f"Before scaling — X2 mean: {X[:,1].mean():.3f},  std: {X[:,1].std():.3f}")
print(f"After  scaling — X2 mean: {X_scaled[:,1].mean():.3f},  std: {X_scaled[:,1].std():.3f}")

---
## 4 · Build the 1-D Prediction Grid

To draw a **smooth fit line**, we cannot just use the training points (they are scattered).  
Instead we create a fine 1-D grid along Feature 1 while **holding Feature 2 fixed at its median**.

This is a standard technique called a **partial dependence slice**: vary one feature,  
freeze the other, and see how the model's prediction changes.


In [ ]:
# 300 evenly-spaced values from min(X1) to max(X1)
grid_f1 = np.linspace(X[:, 0].min(), X[:, 0].max(), 300)

# Feature 2 is held constant at its median across all 300 grid points
grid_f2 = np.full(300, np.median(X[:, 1]))

# Combine into (300, 2) array — same shape as training X
grid_raw = np.column_stack([grid_f1, grid_f2])

# IMPORTANT: use transform (NOT fit_transform) so we use the SAME μ, σ learned above
grid_sc  = scaler.transform(grid_raw)

print(f"grid_raw shape : {grid_raw.shape}")
print(f"grid_sc  shape : {grid_sc.shape}")
print(f"Feature 2 held at median: {np.median(X[:, 1]):.4f}")

---
## 5 · Helper Function: fit_predict(k)

This function encapsulates: train model → predict on grid → compute metrics.  
We call it repeatedly with different K values for comparison.


In [ ]:
def fit_predict(k, metric="euclidean", weights="uniform"):
    """
    Train a KNN Regressor with the given k and return:
      - y_pred_full : predictions on ALL training points (for residuals, metrics)
      - y_pred_line : predictions on the smooth 1-D grid  (for the fit line)
      - r2          : R² score on training data
      - mse         : Mean Squared Error on training data
    """
    model = KNeighborsRegressor(
        n_neighbors = k,
        metric      = metric,   # distance function: euclidean / manhattan / chebyshev
        weights     = weights,  # 'uniform' = equal vote; 'distance' = closer = more weight
    )
    model.fit(X_scaled, y)                     # train: memorise all training points

    y_pred_full = model.predict(X_scaled)      # predict on training X → for metrics
    y_pred_line = model.predict(grid_sc)       # predict on the 300-point grid → smooth line

    r2  = r2_score(y, y_pred_full)
    mse = mean_squared_error(y, y_pred_full)

    return y_pred_full, y_pred_line, r2, mse


# --- quick test ---
K = 5
y_pred_full, y_pred_line, r2_val, mse_val = fit_predict(K)
rmse_val = np.sqrt(mse_val)

print(f"K = {K}")
print(f"R²   = {r2_val:.4f}   (1.0 = perfect fit)")
print(f"MSE  = {mse_val:.4f}")
print(f"RMSE = {rmse_val:.4f}  (same units as y)")

---
## 6 · Plot 1 — Main Fit Line (Feature 1 vs Target)

**What this plot shows:**  
- Blue scatter = actual data points  
- Violet line = KNN's prediction as Feature 1 varies (Feature 2 fixed at median)

**What to observe:**  
- Small K (e.g. K=1): line is very jagged, follows every training point closely → **overfitting**  
- Large K (e.g. K=25): line is very smooth, misses the true pattern → **underfitting**


In [ ]:
# ── Choose K ──────────────────────────────────────────────────────────────────
K = 5   # <-- change this to see different fits
y_pred_full, y_pred_line, r2_val, mse_val = fit_predict(K)

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(PANEL)

# Scatter: raw data points (X[:, 0] = Feature 1 values)
ax.scatter(
    X[:, 0], y,          # x-axis = Feature 1, y-axis = actual target
    color=ACCENT2,
    s=20,                # marker size
    alpha=0.45,          # semi-transparent so the fit line shows through
    zorder=2,            # draw above the grid but below the line
    label="Actual data points"
)

# Fit line: grid_f1 = Feature 1 values, y_pred_line = model predictions at those points
ax.plot(
    grid_f1, y_pred_line,
    color=ACCENT1,
    lw=2.5,              # line width
    zorder=3,            # draw on top of scatter
    label=f"KNN prediction (K={K})"
)

# Optional: vertical dotted line showing where Feature 2 is held fixed
ax.axvline(
    np.median(X[:, 1]),
    color=SUBTEXT, lw=0.8, linestyle=":",
    label=f"Feature 2 median = {np.median(X[:,1]):.2f}"
)

ax.set_title(f"KNN Fit Line  |  K = {K}  |  R² = {r2_val:.4f}", pad=12)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Target  y")
ax.grid(True, alpha=0.35)
ax.legend(fontsize=8, framealpha=0.2)

plt.tight_layout()
plt.show()
print(f"R² = {r2_val:.4f}  |  RMSE = {np.sqrt(mse_val):.4f}")

---
## 7 · Plot 2 — Multi-K Comparison (Overlay)

**What this plot shows:**  
Multiple K values plotted simultaneously with different colours.  
Each line comes from a separately trained KNN model.

**Key idea:**  
By overlaying K=1, 5, 15, 25 on the same axes you can visually see  
the **bias-variance tradeoff** — from wildly overfit (K=1) to over-smooth (K=25).

We use a custom colourmap so each K gets a distinct, visually ordered colour.


In [ ]:
# ── K values to compare ───────────────────────────────────────────────────────
K_VALUES = [1, 3, 5, 10, 15, 25]   # <-- add or remove values freely

# ── Custom colourmap: orange → violet → blue → green ─────────────────────────
# LinearSegmentedColormap.from_list creates a smooth gradient between given colours.
# We then sample evenly-spaced colours from it — one per K value.
cmap_k = LinearSegmentedColormap.from_list(
    "kmap", ["#f97316", "#a78bfa", "#60a5fa", "#34d399"]
)

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(PANEL)

# Background scatter (light alpha so the lines stay visible)
ax.scatter(X[:, 0], y, color=ACCENT2, s=14, alpha=0.25, zorder=2, label="Data")

for i, k_c in enumerate(K_VALUES):
    # Sample colour: i / (total-1) maps index to [0, 1] range for the colourmap
    colour = cmap_k(i / max(len(K_VALUES) - 1, 1))

    _, pred_line_c, r2_c, _ = fit_predict(k_c)

    ax.plot(
        grid_f1, pred_line_c,
        color=colour,
        lw=2.2,
        alpha=0.9,
        label=f"K={k_c}   R²={r2_c:.3f}"
    )

ax.set_title("Multi-K Comparison — how K changes the fit", pad=12)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Target  y")
ax.grid(True, alpha=0.35)
ax.legend(fontsize=8, framealpha=0.2, ncol=2)

plt.tight_layout()
plt.show()

---
## 8 · Plot 3 — Residuals Plot

**What are residuals?**  
`Residual = Actual y − Predicted ŷ`

**What this plot shows:**  
- x-axis: predicted value ŷ  
- y-axis: residual (how wrong the prediction was)

**What a good model looks like:**  
- Points scattered randomly around the **zero line** with no pattern  
- No funnel/fan shape (that would signal heteroscedasticity)

**The ±1σ band** shades the region within one standard deviation of zero.  
~68% of residuals should fall inside if errors are Gaussian.


In [ ]:
# ── Use K=5 predictions already computed ──────────────────────────────────────
K = 5
y_pred_full, _, r2_val, mse_val = fit_predict(K)

residuals = y - y_pred_full          # actual minus predicted for every training point
sigma_res = np.std(residuals)        # std of residuals = spread of errors

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(PANEL)

# Zero reference line — residuals should scatter around this
ax.axhline(
    0,
    color=ACCENT4,
    lw=1.5,
    linestyle="--",
    alpha=0.8,
    label="Zero residual line"
)

# ±1σ shaded band
# axhspan shades a horizontal strip from ymin to ymax across the entire x-range
ax.axhspan(
    -sigma_res, sigma_res,
    color=ACCENT3,
    alpha=0.08,
    label=f"±1σ band  (σ = {sigma_res:.3f})"
)

# Scatter: predicted ŷ on x-axis, residual on y-axis
ax.scatter(
    y_pred_full, residuals,
    color=ACCENT3,
    s=20,
    alpha=0.55,
    zorder=3
)

ax.set_title(f"Residuals Plot  |  K = {K}  |  RMSE = {np.sqrt(mse_val):.4f}", pad=12)
ax.set_xlabel("Predicted  ŷ")
ax.set_ylabel("Residual  (y − ŷ)")
ax.grid(True, alpha=0.35)
ax.legend(fontsize=8, framealpha=0.2)

plt.tight_layout()
plt.show()

print(f"Mean of residuals  : {residuals.mean():.6f}   (should be ~0 for KNN)")
print(f"Std  of residuals  : {sigma_res:.4f}")

---
## 9 · Plot 4 — R² Score vs K Curve

**What this plot shows:**  
How training R² changes as K goes from 1 to 30.

**What to expect:**  
- K=1: perfect R²=1.0 on training data (every point predicts itself — pure overfitting)  
- As K increases: R² drops (model smooths out and can no longer memorise training data)  
- Very large K: R² plateaus at a low value (underfitting, high bias)

The vertical orange line marks your currently selected K.

> ⚠️ **This is training R² only.** In practice you would also plot a validation/test curve  
> and the best K is where the two curves diverge the least.


In [ ]:
K_SELECTED = 5   # mark this K on the plot

# ── Compute R² for every K from 1 to 30 ──────────────────────────────────────
k_range  = np.arange(1, 31)
r2_train = []

for k_iter in k_range:
    _, _, r2_iter, _ = fit_predict(k_iter)
    r2_train.append(r2_iter)

r2_selected = r2_train[K_SELECTED - 1]   # index = K - 1

# ── Draw ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.set_facecolor(PANEL)

# Main R² curve with circular markers at each integer K
ax.plot(
    k_range, r2_train,
    color=ACCENT1,
    lw=2.2,
    marker="o",
    markersize=5,
    alpha=0.85,
    label="Training R²"
)

# Vertical dashed line for selected K
ax.axvline(
    K_SELECTED,
    color=ACCENT4,
    lw=1.8,
    linestyle="--",
    label=f"Selected  K = {K_SELECTED}"
)

# Larger highlighted dot at the selected K's R² value
ax.scatter(
    [K_SELECTED], [r2_selected],
    color=ACCENT4,
    s=100,          # bigger marker
    zorder=5,
    label=f"R² at K={K_SELECTED} = {r2_selected:.4f}"
)

ax.set_title("R² Score vs K  (Training Set)", pad=12)
ax.set_xlabel("K  (number of neighbours)")
ax.set_ylabel("R²  Score")
ax.set_xticks(range(1, 31, 2))     # label every other K to avoid crowding
ax.grid(True, alpha=0.35)
ax.legend(fontsize=8, framealpha=0.2)

plt.tight_layout()
plt.show()

print("\nR² for selected K values:")
for k_show in [1, 3, 5, 10, 15, 20, 25, 30]:
    print(f"  K={k_show:2d}  →  R² = {r2_train[k_show-1]:.4f}")

---
## 10 · Full 2×2 Dashboard (all 4 plots together)

This is exactly the layout used in the Streamlit app.  
`GridSpec(2, 2)` divides the figure into a 2-row × 2-column grid,  
and `fig.add_subplot(gs[row, col])` places each axis in a cell.


In [ ]:
K = 5   # change this to see a different K in the full dashboard

y_pred_full, y_pred_line, r2_val, mse_val = fit_predict(K)
rmse_val = np.sqrt(mse_val)

# ── Figure + GridSpec ─────────────────────────────────────────────────────────
# hspace / wspace control vertical / horizontal spacing between subplots
# left/right/top/bottom are figure margins as fractions of figure size
fig = plt.figure(figsize=(16, 12), facecolor=BG)
gs  = gridspec.GridSpec(
    2, 2, figure=fig,
    hspace=0.38, wspace=0.32,
    left=0.06, right=0.97, top=0.94, bottom=0.07
)

# ──────────────────────────────────────────────────────────────────────────────
# SUBPLOT 1 (top-left):  Main fit line
# ──────────────────────────────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])   # row 0, column 0
ax1.scatter(X[:, 0], y, color=ACCENT2, s=18, alpha=0.45, zorder=2, label="Data points")
ax1.plot(grid_f1, y_pred_line, color=ACCENT1, lw=2.5, zorder=3, label=f"KNN fit (K={K})")
ax1.axvline(np.median(X[:, 1]), color=SUBTEXT, lw=0.5, linestyle=":")
ax1.set_title(f"Fit Line — Feature 1 vs Target  |  K = {K}", pad=10)
ax1.set_xlabel("Feature 1"); ax1.set_ylabel("Target y")
ax1.grid(True, alpha=0.4); ax1.legend(fontsize=7.5, framealpha=0.2)

# ──────────────────────────────────────────────────────────────────────────────
# SUBPLOT 2 (top-right):  Multi-K comparison
# ──────────────────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])   # row 0, column 1
ax2.scatter(X[:, 0], y, color=ACCENT2, s=14, alpha=0.25, zorder=2, label="Data")
cmap_k = LinearSegmentedColormap.from_list("kmap", ["#f97316", "#a78bfa", "#60a5fa", "#34d399"])
for i, k_c in enumerate([1, 5, 15, 25]):
    clr = cmap_k(i / 3)
    _, pred_c, r2_c, _ = fit_predict(k_c)
    ax2.plot(grid_f1, pred_c, color=clr, lw=2.2, alpha=0.9, label=f"K={k_c} R²={r2_c:.3f}")
ax2.set_title("Multi-K Comparison", pad=10)
ax2.set_xlabel("Feature 1"); ax2.set_ylabel("Target y")
ax2.grid(True, alpha=0.4); ax2.legend(fontsize=7, framealpha=0.2, ncol=2)

# ──────────────────────────────────────────────────────────────────────────────
# SUBPLOT 3 (bottom-left):  Residuals
# ──────────────────────────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])   # row 1, column 0
residuals = y - y_pred_full
sigma = np.std(residuals)
ax3.axhline(0, color=ACCENT4, lw=1.2, linestyle="--", alpha=0.7)
ax3.axhspan(-sigma, sigma, color=ACCENT3, alpha=0.06, label=f"±1σ = {sigma:.2f}")
ax3.scatter(y_pred_full, residuals, color=ACCENT3, s=18, alpha=0.55, zorder=2)
ax3.set_title(f"Residuals (Actual − Predicted)  |  K = {K}", pad=10)
ax3.set_xlabel("Predicted ŷ"); ax3.set_ylabel("Residual")
ax3.grid(True, alpha=0.4); ax3.legend(fontsize=7.5, framealpha=0.2)

# ──────────────────────────────────────────────────────────────────────────────
# SUBPLOT 4 (bottom-right):  R² vs K curve
# ──────────────────────────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])   # row 1, column 1
k_range = np.arange(1, 31)
r2_train = [fit_predict(ki)[2] for ki in k_range]    # list comprehension: R² for each K
ax4.plot(k_range, r2_train, color=ACCENT1, lw=2.2, marker="o", markersize=4, alpha=0.85, label="Train R²")
ax4.axvline(K, color=ACCENT4, lw=1.5, linestyle="--", label=f"Selected K={K}")
ax4.scatter([K], [r2_val], color=ACCENT4, s=80, zorder=5)
ax4.set_title("R² Score vs K  (Training Set)", pad=10)
ax4.set_xlabel("K  (number of neighbours)"); ax4.set_ylabel("R² Score")
ax4.set_xticks(range(1, 31, 2)); ax4.grid(True, alpha=0.4)
ax4.legend(fontsize=7.5, framealpha=0.2)

# ── Super-title ───────────────────────────────────────────────────────────────
fig.suptitle(
    f"KNN Regressor Dashboard   ·   K = {K}   ·   R² = {r2_val:.4f}   ·   RMSE = {rmse_val:.4f}",
    fontsize=12, fontweight="bold", color=TEXT, y=0.975
)

plt.show()

---
## 11 · Experiment: Loop over K values

Run this cell to quickly scan how metrics change across every K from 1 to 20.


In [ ]:
print(f"{'K':>4}  {'R²':>8}  {'RMSE':>8}  {'MSE':>10}  Interpretation")
print("-" * 65)
for k_i in range(1, 21):
    yp, _, r2_i, mse_i = fit_predict(k_i)
    rmse_i = np.sqrt(mse_i)
    if   k_i <= 2:   note = "⚠️  likely overfitting"
    elif k_i <= 8:   note = "✅  balanced"
    elif k_i <= 15:  note = "🔵 smoothing out"
    else:            note = "⛔ underfitting risk"
    print(f"{k_i:>4}  {r2_i:>8.4f}  {rmse_i:>8.4f}  {mse_i:>10.4f}  {note}")

---
## 12 · Summary: What we learned

| Concept | Explanation |
|---|---|
| **KNN Regression** | Predicts by averaging the target values of the K nearest training points |
| **Distance metric** | How "nearest" is measured (Euclidean by default) |
| **StandardScaler** | Required before KNN so feature scales don't distort distances |
| **1-D slice / partial dependence** | Fix Feature 2 at median, vary Feature 1 → draws a clean fit line |
| **Small K** | Follows training data closely → high variance → jagged line → overfits |
| **Large K** | Averages over many points → high bias → flat/smooth line → underfits |
| **R² score** | Fraction of variance explained; 1.0 = perfect, 0 = no better than mean |
| **Residual plot** | Random scatter around zero = good model; patterns = problem |
| **R² vs K curve** | K=1 always gives train R²=1 (memorisation); use validation set to pick optimal K |

### Next steps
- Add a train/test split and plot **validation R²** alongside training R² to find the true optimal K
- Try `weights="distance"` in the sidebar (closer neighbours vote more)
- Try other distance metrics: `manhattan`, `chebyshev`
